# HSIP Data Build - Windows Notebook

This notebook runs the data-build pipeline from Windows Python.

It can rebuild:

- `jurisdictions-<hash>.geojson`
- `app-<hash>.db` crash SQLite artifact
- `crashes_kab-<hash>.pmtiles`
- overlay PMTiles such as `roads-<hash>.pmtiles` and `high_injury_network-<hash>.pmtiles`
- `manifest.json`

PMTiles generation uses `tippecanoe` when it is available. On Windows without WSL, the pipeline falls back to the PMTiles writer bundled with GDAL through `pyogrio`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path(r"C:/App/hisp_code/tools/data-build")]
DATA_BUILD_DIR = next((p.resolve() for p in candidates if (p / "build.py").exists()), None)
if DATA_BUILD_DIR is None:
    raise FileNotFoundError("Could not find tools/data-build/build.py")

REPO_ROOT = DATA_BUILD_DIR.parent.parent
INPUT_DIR = REPO_ROOT / "input_data"
OUTPUT_DIR = REPO_ROOT / "public"

sys.path.insert(0, str(DATA_BUILD_DIR))

print(f"Data build dir: {DATA_BUILD_DIR}")
print(f"Repo root:       {REPO_ROOT}")
print(f"Input dir:       {INPUT_DIR}")
print(f"Output dir:      {OUTPUT_DIR}")

In [ ]:
required_modules = ["geopandas", "pandas", "yaml", "shapely", "requests", "pyogrio"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

if missing:
    print("Missing modules:", missing)
    print("Run the next cell to install them into this notebook kernel, then restart the kernel.")
else:
    import geopandas as gpd
    import pandas as pd
    import pyogrio
    import shapely
    print("Dependency check passed")
    print("geopandas", gpd.__version__)
    print("pyogrio", pyogrio.__version__)
    print("GDAL", pyogrio.__gdal_version__)
    drivers = pyogrio.list_drivers()
    print("PMTiles driver:", drivers.get("PMTiles"))
    if drivers.get("PMTiles") != "rw":
        raise RuntimeError("GDAL/pyogrio does not report PMTiles read/write support")

## Install Missing Notebook Dependencies

Run this cell only if the dependency check above reports missing modules. It installs into the active notebook kernel, which is what VS Code uses when it runs notebook cells.

In [ ]:
requirements_path = DATA_BUILD_DIR / "requirements.txt"
print("Installing from", requirements_path)
%pip install -r "{requirements_path}"
print("Restart the notebook kernel after this cell finishes, then rerun from the top.")

## PMTiles Backend Check

The pipeline uses `tippecanoe` when it is on PATH. If not, `output/pmtiles.py` writes PMTiles through GeoPandas/pyogrio/GDAL. This smoke test creates and reads a tiny PMTiles file so the fallback is checked before a long build.

In [ ]:
from shapely.geometry import Point

backend = "tippecanoe" if shutil.which("tippecanoe") else "GDAL/pyogrio fallback"
print("PMTiles backend:", backend)

smoke_path = DATA_BUILD_DIR / "output" / "notebook_pmtiles_smoke.pmtiles"
smoke_path.parent.mkdir(parents=True, exist_ok=True)
smoke_path.unlink(missing_ok=True)

smoke_gdf = gpd.GeoDataFrame(
    {"id": [1], "name": ["Houston"]},
    geometry=[Point(-95.3698, 29.7604)],
    crs="EPSG:4326",
)
smoke_gdf.to_file(
    smoke_path,
    driver="PMTiles",
    layer="smoke",
    MINZOOM=0,
    MAXZOOM=13,
    MAX_SIZE=500000,
)

print("Wrote", smoke_path, smoke_path.stat().st_size, "bytes")
print("Layers:", pyogrio.list_layers(smoke_path).tolist())
print("Info:", pyogrio.read_info(smoke_path)["layer_name"])
smoke_path.unlink(missing_ok=True)

In [ ]:
import pyogrio

required_inputs = [
    INPUT_DIR / "cris_export",
    INPUT_DIR / "hgac_roads.gpkg",
    INPUT_DIR / "HGAC HIN.gpkg",
    INPUT_DIR / "jurisdictions.gpkg",
]

for path in required_inputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {path}")

print("\nGeoPackage layers")
for name in ["hgac_roads.gpkg", "HGAC HIN.gpkg", "jurisdictions.gpkg"]:
    path = INPUT_DIR / name
    print(name, pyogrio.list_layers(path).tolist())

print("\nCRIS year folders")
for folder in sorted((INPUT_DIR / "cris_export").iterdir()):
    if not folder.is_dir():
        continue
    files = list(folder.glob("*.csv"))
    crash = [p for p in files if "_crash_" in p.name.lower()]
    unit = [p for p in files if "_unit_" in p.name.lower()]
    person = [p for p in files if "_person_" in p.name.lower() and "_primaryperson_" not in p.name.lower()]
    primary = [p for p in files if "_primaryperson_" in p.name.lower()]
    mb = sum(p.stat().st_size for p in files) / (1024 * 1024)
    print(
        f"{folder.name}: crash={len(crash)}, unit={len(unit)}, "
        f"person={len(person)}, primaryperson={len(primary)}, "
        f"csv={len(files)}, {mb:.1f} MB"
    )

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", ".", "-p", "test*.py"],
    cwd=DATA_BUILD_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Unit tests failed")

In [ ]:
from config import load_build_config, derive_ea_ids, derive_hsip_fields

CONFIG_PATH = DATA_BUILD_DIR / "build-config.yaml"
HSIP_CONFIG_DIR = REPO_ROOT / "config" / "hsip"

cfg = load_build_config(CONFIG_PATH)
ea_ids = derive_ea_ids(HSIP_CONFIG_DIR)
hsip_fields = derive_hsip_fields(HSIP_CONFIG_DIR)
overlay_cfgs = cfg.get("overlays", {})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Flag fields: {len(ea_ids)} EA, {len(hsip_fields)} HSIP")
print("Overlays:", list(overlay_cfgs))

## Rebuild Jurisdictions

This cell rebuilds the jurisdiction GeoJSON from `input_data/jurisdictions.gpkg`, publishes it under a content-hashed filename, and keeps all untouched artifacts from the current manifest.

In [ ]:
import json
from ingest import jurisdictions as jurisdiction_ingest
from output.publish import publish, stable_path

source = cfg["jurisdictions"]["source"]
unincorp = cfg["jurisdictions"].get("unincorporated_city_id")

if source["type"] != "gpkg":
    raise ValueError("This Windows notebook cell expects local GPKG jurisdictions")

jurisdictions = jurisdiction_ingest.ingest_from_gpkg(
    REPO_ROOT / source["path"],
    source.get("layer"),
    unincorporated_city_id=unincorp,
)

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"id": j["id"], "name": j["name"], "jurisdictionType": j["type"]},
            "geometry": j["geometry"],
        }
        for j in jurisdictions
    ],
}

out_path = stable_path(OUTPUT_DIR, "jurisdictions")
with open(out_path, "w") as f:
    json.dump(geojson, f)

print(f"Wrote {len(jurisdictions)} jurisdictions to {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
manifest = publish(OUTPUT_DIR, {"jurisdictions"}, {name: None for name in overlay_cfgs}, crash_years=None)
manifest

## Rebuild Crashes

This cell rebuilds both crash artifacts through the original pipeline:

- `app-<hash>.db`
- `crashes_kab-<hash>.pmtiles`

If `tippecanoe` is not available, crash PMTiles are written by the GDAL/pyogrio fallback.

In [ ]:
result = subprocess.run(
    [sys.executable, "build.py", "--only", "crashes"],
    cwd=DATA_BUILD_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Crash build failed")

## Rebuild One Overlay

Set `LAYER_TO_BUILD` to an overlay name from `build-config.yaml`. The current overlays are `roads` and `high_injury_network`.

In [ ]:
LAYER_TO_BUILD = "roads"  # or "high_injury_network"

if LAYER_TO_BUILD not in overlay_cfgs:
    raise ValueError(f"Unknown overlay {LAYER_TO_BUILD!r}; known overlays: {list(overlay_cfgs)}")

result = subprocess.run(
    [sys.executable, "build.py", "--only", LAYER_TO_BUILD],
    cwd=DATA_BUILD_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Overlay build failed: {LAYER_TO_BUILD}")

## Optional Full Build

This calls the original CLI build. It uses `tippecanoe` when available, otherwise it uses the GDAL PMTiles fallback.

In [ ]:
if shutil.which("tippecanoe") is None:
    print("tippecanoe is not available on PATH. The build will use GDAL PMTiles fallback.")

result = subprocess.run(
    [sys.executable, "build.py"],
    cwd=DATA_BUILD_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Full build failed")

In [ ]:
manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

print("\nPublished PMTiles")
published = {
    **manifest.get("artifacts", {}),
    **{k: v["file"] for k, v in manifest.get("overlays", {}).items()},
}
for key, filename in published.items():
    if not filename.endswith(".pmtiles"):
        continue
    path = OUTPUT_DIR / filename
    info = pyogrio.read_info(path)
    metadata = info.get("dataset_metadata") or {}
    print(
        f"{key}: {filename} | layer={info['layer_name']} | "
        f"features={info['features']:,} | zoom={metadata.get('ZOOM_LEVEL')} | "
        f"size={path.stat().st_size / 1e6:.1f} MB"
    )